# The Charging Recommendation Engine

Module G ties the platform together. A driver gives their situation — vehicle, current
and target state of charge, distance to cover, when they can plug in and for how long —
and the engine returns a concrete plan: which charger, when to start, and the expected
energy, time and cost, with a one-line reason.

## Why this engine is built on physics, not the ML models

Notebooks 03 and 04 established that on this dataset the energy and duration regressors
**do not beat a mean baseline** — the synthetic targets carry no signal. A recommendation
engine that leaned on them would tell every driver the same thing.

So the estimates here come from **charging physics**, which holds regardless of the data:

$$\text{energy} = \frac{\text{SOC}_\text{target} - \text{SOC}_\text{start}}{100}
\times \text{capacity} \div \eta \qquad
\text{time} = \frac{\text{energy}}{\text{power}(\text{charger})} \qquad
\text{cost} = \text{energy} \times \text{price}$$

where $\eta \approx 0.9$ is the charging efficiency (a little more is drawn than reaches
the battery), $\text{power}(\text{charger})$ is the charger's nominal kW, and
$\text{price}$ is calibrated from the dataset (mean cost ÷ mean energy ≈ \$0.53/kWh).

The trained models are still consulted as a **sanity band** — if the energy regressor's
population-average prediction is wildly different from the physics number, the engine
says so — and the demand forecaster picks the quietest hour to start.

## Contents

* The pure estimator functions
* Choosing the charger and the start time
* Wiring in the trained artifacts
* Three worked scenarios
* Limitations

## The pure estimator functions

Each estimator takes plain numbers and returns a plain number, so it is easy to test and
to reason about. They live in `evcharging.recommendation.strategy`.

In [1]:
import sys

sys.path.insert(0, "../src")

import pandas as pd

from evcharging.recommendation.strategy import (
    estimate_cost_usd,
    estimate_duration_hours,
    estimate_energy_kwh,
    nominal_power_kw,
)

# A Tesla Model 3 (60 kWh) going from 30% to 80%
energy = estimate_energy_kwh(soc_start_pct=30, soc_target_pct=80, battery_capacity_kwh=60)
energy

33.333333333333336

In [2]:
# The nominal power assumptions, in kW
{c: nominal_power_kw(c) for c in ["Level 1", "Level 2", "DC Fast Charger"]}

{'Level 1': 1.4, 'Level 2': 7.0, 'DC Fast Charger': 50.0}

In [3]:
# Time to deliver that energy on each charger type
{c: round(estimate_duration_hours(energy, c), 2) for c in ["Level 1", "Level 2", "DC Fast Charger"]}

{'Level 1': 23.81, 'Level 2': 4.76, 'DC Fast Charger': 0.67}

In [4]:
# Cost at the calibrated price
round(estimate_cost_usd(energy), 2)

17.67

For this half-charge of a 60 kWh pack the engine expects **~33 kWh**, which is minutes on
a DC fast charger, about 5 hours on Level 2, or an overnight job on Level 1, costing
about **\$18**.

## Choosing the charger and the start time

**Charger choice.** Slower charging is cheaper and gentler on the battery, so the engine
picks the *slowest* charger that still finishes inside the driver's time budget. Only when
the budget is tight does it step up to Level 2, then DC fast.

In [5]:
from evcharging.recommendation.strategy import recommend_charger_type

for hours in [12, 5, 2, 0.5]:
    charger, why = recommend_charger_type(energy, hours_available=hours)
    print(f"{hours:>4} h available -> {charger:<16} ({why})")

  12 h available -> Level 2          (Level 2 is the gentlest charger that fits the time available)
   5 h available -> Level 2          (Level 2 is the gentlest charger that fits the time available)
   2 h available -> DC Fast Charger  (DC fast charging is needed to finish within the time available)
 0.5 h available -> DC Fast Charger  (even DC fast charging will not fully finish in the time available)


**Start time.** `best_charging_window` slides the charging window over the next 24 hours
and picks the start hour with the lowest total predicted network demand, so the driver
charges when the grid is quietest.

In [6]:
from evcharging.recommendation.strategy import best_charging_window

# a demand curve with a clear overnight trough
demo_demand = pd.Series({h: 100.0 for h in range(24)})
demo_demand.loc[[1, 2, 3, 4, 5]] = 20.0

best_charging_window(demo_demand, duration_hours=3.0, earliest_hour=18)

(1, 4)

Given a window that can start any time from 18:00 and a 3-hour charge, the engine points
the driver at the small hours of the morning, where the demo demand curve bottoms out.

## Wiring in the trained artifacts

`load_context` loads the three artifacts the engine can use — the session segmenter (for
a behavioural archetype label), the energy regressor (for the sanity band), and a
demand-by-hour curve derived from the demand forecaster. Any that are missing come back
as `None` and the engine still works.

In [7]:
from evcharging.recommendation.strategy import load_context

context = {k: v is not None for k, v in load_context().items()}
context

{'segmenter_bundle': True, 'energy_model': True, 'demand_by_hour': True}

In [8]:
ctx = load_context()
ctx["demand_by_hour"].round(1)

0     41.7
1     41.8
2     44.7
3     43.6
4     44.1
5     41.2
6     41.7
7     40.5
8     42.1
9     43.4
10    43.0
11    43.7
12    43.6
13    42.7
14    42.7
15    43.5
16    43.5
17    42.9
18    44.2
19    42.3
20    43.7
21    42.7
22    43.6
23    43.5
dtype: float64

The demand curve is nearly flat — the forecaster learned the mean, as notebook 07
explained — so the "quietest hour" is only weakly meaningful here. On a real network with
a genuine daily load shape this is where the recommendation would have real teeth.

## Three worked scenarios

In [9]:
from evcharging.recommendation import RecommendationRequest, recommend

ctx = load_context()


def show(title, request):
    rec = recommend(
        request,
        demand_by_hour=ctx["demand_by_hour"],
        segmenter_bundle=ctx["segmenter_bundle"],
        energy_model=ctx["energy_model"],
    )
    print(f"### {title}")
    for key, value in rec.as_dict().items():
        print(f"  {key:26} {value}")
    print()

In [10]:
# 1. Commuter, home overnight, plenty of time
show("Commuter — overnight top-up", RecommendationRequest(
    vehicle_model="Nissan Leaf", battery_capacity_kwh=40,
    soc_start_pct=35, soc_target_pct=90, distance_km=45,
    earliest_hour=20, hours_available=10, user_type="Commuter",
))

### Commuter — overnight top-up
  recommended_charger        Level 2
  estimated_energy_kwh       24.44
  estimated_duration_hours   3.49
  estimated_cost_usd         12.96
  charging_window            05:00-09:00
  session_archetype          Long slow low-energy sessions
  reason                     You need about 24 kWh to go from 35% to 90%. Level 2 is the gentlest charger that fits the time available. Start around 05:00, when predicted network demand is lowest.
  model_energy_kwh           42.64
  notes                      ['the trained energy model expects ~43 kWh for a session like this (population average); the physics estimate is used above']



In [11]:
# 2. Long-distance driver, quick stop mid-trip
show("Long-distance traveller — fast highway stop", RecommendationRequest(
    vehicle_model="Hyundai Kona", battery_capacity_kwh=64,
    soc_start_pct=15, soc_target_pct=80, distance_km=320,
    earliest_hour=13, hours_available=1, user_type="Long-Distance Traveler",
))

### Long-distance traveller — fast highway stop
  recommended_charger        DC Fast Charger
  estimated_energy_kwh       46.22
  estimated_duration_hours   0.92
  estimated_cost_usd         24.5
  charging_window            07:00-08:00
  session_archetype          Long fast low-energy sessions
  reason                     You need about 46 kWh to go from 15% to 80%. DC fast charging is needed to finish within the time available. Start around 07:00, when predicted network demand is lowest.
  model_energy_kwh           42.64
  notes                      []



In [12]:
# 3. Casual driver, small daytime charge
show("Casual driver — short daytime charge", RecommendationRequest(
    vehicle_model="BMW i3", battery_capacity_kwh=42,
    soc_start_pct=55, soc_target_pct=70, distance_km=20,
    earliest_hour=10, hours_available=4, user_type="Casual Driver",
))

### Casual driver — short daytime charge
  recommended_charger        Level 2
  estimated_energy_kwh       7.0
  estimated_duration_hours   1.0
  estimated_cost_usd         3.71
  charging_window            07:00-08:00
  session_archetype          Long slow low-energy sessions
  reason                     You need about 7 kWh to go from 55% to 70%. Level 2 is the gentlest charger that fits the time available. Start around 07:00, when predicted network demand is lowest.
  model_energy_kwh           42.64
  notes                      ['the trained energy model expects ~43 kWh for a session like this (population average); the physics estimate is used above']



**Observations**

* The **commuter** with 10 hours at home is steered to a slow, cheap Level 1 or Level 2
  charge overnight — the engine does not waste a fast charger when time is abundant.
* The **long-distance traveller** with one hour gets a DC fast charger; if even that
  cannot finish the 65-point SOC swing in an hour, the notes say to expect a partial
  charge.
* The **casual driver's** small 15-point top-up is quick on any charger, so it again
  picks the gentlest one that fits.
* Every recommendation carries a plain-language `reason` and a behavioural `archetype`
  from the segmenter.

## Comparing charger options

The engine also exposes `compare_chargers`, which returns the same charge costed on all
three charger types. The energy and cost are identical (they depend only on the SOC
swing); only the time differs. This is the "your options" table a driver-facing UI would
show next to the headline recommendation.

In [13]:
from evcharging.recommendation import compare_chargers

req = RecommendationRequest(
    vehicle_model="Hyundai Kona", battery_capacity_kwh=64,
    soc_start_pct=20, soc_target_pct=80, hours_available=3,
)
pd.DataFrame(compare_chargers(req))

,charger_type,power_kw,energy_kwh,duration_hours,cost_usd,fits_time_budget
0,Level 1,1.4,42.67,30.48,22.61,False
1,Level 2,7.0,42.67,6.10,22.61,False
2,DC Fast Charger,50.0,42.67,0.85,22.61,True


For this 60-point charge of a 64 kWh pack the driver needs ~43 kWh at ~\$23 whichever
charger they pick. Level 1 would take over a day; Level 2 fits an afternoon; DC fast is
under an hour. With a 3-hour budget the engine picks Level 2 — the slowest option in the
`fits_time_budget` column.

`recommend_batch` runs the engine over many requests with one shared context (artifacts
loaded once), for populating a dashboard or scoring a scenario grid.

In [14]:
from evcharging.recommendation import recommend_batch

targets = [40, 60, 80, 100]
grid = [
    RecommendationRequest(vehicle_model="Tesla Model 3", battery_capacity_kwh=60,
                          soc_start_pct=20, soc_target_pct=t, hours_available=6)
    for t in targets
]
recs = recommend_batch(grid, **{k: ctx[k] for k in
    ("demand_by_hour", "segmenter_bundle", "energy_model")})

pd.DataFrame({
    "target_soc_pct": targets,
    "charger": [r.recommended_charger for r in recs],
    "energy_kwh": [round(r.estimated_energy_kwh, 1) for r in recs],
    "duration_h": [round(r.estimated_duration_hours, 2) for r in recs],
    "cost_usd": [round(r.estimated_cost_usd, 2) for r in recs],
})

,target_soc_pct,charger,energy_kwh,duration_h,cost_usd
0,40,Level 2,13.3,1.90,7.07
1,60,Level 2,26.7,3.81,14.13
2,80,Level 2,40.0,5.71,21.20
3,100,DC Fast Charger,53.3,1.07,28.27


## Limitations

* **Estimates are physics, not learned.** They assume a constant charging power (real
  sessions taper near full) and a flat price (real tariffs vary by time and location).
  They are good planning numbers, not guarantees.
* **The demand-based start time is weak on this data** because the forecast is nearly
  flat. The mechanism is correct and would matter on a real load curve.
* **No live availability.** The engine does not know whether a specific charger is free;
  it recommends a *type* and a *time*, not a station.
* **The archetype is descriptive** (silhouette ≈ 0.12, notebook 05) and is used only for
  context in the reason string, never to override the physics.

Despite the data, the engine is a complete decision layer: it takes the same inputs a
real product would, combines physics with the trained models and the demand forecast, and
returns an actionable, explained plan. Phase 4 exposes it as `POST /recommend`.

## Summary / Key Takeaways

* The recommendation engine estimates energy, duration and cost from **charging physics**
  because the ML regressors do not beat a mean on this dataset.
* It recommends the **slowest charger that fits the time budget** and the **quietest hour
  to start**, and consults the trained models as a sanity band and for a behavioural
  archetype.
* All estimator functions are pure and unit-tested; `recommend` wires them to the
  artifacts via `load_context`. `compare_chargers` returns the same charge on all three
  charger types; `recommend_batch` runs many requests with one shared context.
* This completes Phases 1–3 — the data pipeline, five models, and the decision layer.